# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminamandzukic/FlyRank_starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is primarily a **ranking/scoring task**. The goal is to assign each content page a priority score and rank pages according to which ones should receive human review first.

A binary classification label such as whether a page is declining can be useful as a proxy during model development, but the final decision is not simply whether a page belongs to one class. A content or SEO reviewer has limited time and needs an ordered queue of pages so that the most promising review candidates appear near the top.

The ML output should support the decision: **which pages should be reviewed first for possible refresh, expansion, protection, pruning, or monitoring?**


In [12]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")

Rows: 30,000
Unique content items: 30,000
Unique clients: 32


## 2. Target or proxy

For the starter dataset, I will use **`is_declining_label` as a provisional proxy target**. This variable indicates whether a content item's recent trend is classified as downward.

The proxy is useful because pages showing decline may deserve reviewer attention, but it is not the same as the true outcome I ultimately care about. A downward trend does not automatically mean that a page should be refreshed, and the label is derived from the same current observation window rather than from a later observed outcome.

A stronger target for a future version of the project would use separate time windows, for example:

**features from the previous 90 days - whether the page meaningfully declines, recovers, or presents an opportunity during the following 30 days.**

For this notebook, I will therefore treat `is_declining_label` as a **proxy for review priority**, not as ground truth that a page requires a content change.


In [13]:
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Creating the provisional proxy target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Inspecting the target distribution
n_declining = df["is_declining_label"].sum()
pct_declining = 100 * df["is_declining_label"].mean()

print(f"Declining pages: {n_declining:,}")
print(f"Share declining: {pct_declining:.1f}%")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

Declining pages: 16,262
Share declining: 54.2%

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

I will use **Precision@K** as the primary success metric.

The purpose of the model is to rank pages so that a reviewer can inspect the most promising candidates first. Because reviewer capacity is limited, performance near the top of the ranked list matters more than overall classification accuracy.

Precision@K measures the proportion of pages in the top K recommendations that match the proxy target. 

A higher Precision@K means that reviewer time is concentrated on pages that are more likely to deserve attention according to the current proxy. For this exercise, I will use **Precision@50** as a concrete metric, while treating the decline label as a provisional proxy.

In [14]:
K = 50

# a variable that shows the maximum possible Precision@50 
# (in case where the hypothetical model ranks all declining pages first) 
top_k_example = df.sort_values( 
    "is_declining_label",
    ascending=False
).head(K)

precision_at_k = top_k_example["is_declining_label"].mean()

print(f"K = {K}")
print(f"Example Precision@{K}: {precision_at_k:.2f}")
print(f"Overall declining rate: {df['is_declining_label'].mean():.2f}")

K = 50
Example Precision@50: 1.00
Overall declining rate: 0.54


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content item (page) per row**.

Each row contains observable information about a single pseudonymized content item, including search performance, engagement, content metadata, and other measurements from the trailing observation window.

The model would assign a priority score to each content item, and reviewers would then inspect the highest-ranked pages first.


In [15]:
columns_to_show = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "is_declining_label"
]

df[columns_to_show].head(10)


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,down,1


In [16]:
print(f"Rows: {len(df):,}")
print(f"Unique content items: {df['content_id'].nunique():,}")

if len(df) == df["content_id"].nunique():
    print("Verified: one row corresponds to one unique content item.")
else:
    print("Warning: some content items appear more than once.")

Rows: 30,000
Unique content items: 30,000
Verified: one row corresponds to one unique content item.


## 5. Why ML beats a fixed rule here

A fixed rule may be useful as a baseline, but it is unlikely to capture the full prioritization problem well.

Whether a page deserves review can depend on several signals at the same time, such as search impressions, clicks, CTR, average position, engagement, content characteristics, and recent performance patterns. The importance of one signal may also depend on the values of others. 

A hand-written rule would require manually choosing thresholds and combinations of these signals. Those thresholds may be arbitrary and may not generalize well across different content items or clients.

A machine-learning scoring model can instead learn how multiple observable signals combine to rank pages. The model should still be compared against a simple fixed-rule baseline, and ML is only justified if it produces a more useful top-ranked review queue.

In [17]:
rule_thresholds = [100, 500, 1000, 5000]

results = []

for threshold in rule_thresholds:
    selected = df[df["impressions_90d"] >= threshold]

    results.append({
        "impression_threshold": threshold,
        "pages_selected": len(selected),
        "declining_rate": selected["is_declining_label"].mean()
    })

pd.DataFrame(results)

,impression_threshold,pages_selected,declining_rate
0,100,22006,0.597655
1,500,16726,0.595540
2,1000,13512,0.594361
3,5000,6151,0.546740


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.